###GOLD LAYER

In [0]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GoldLayer")

####Reading Silver Table

In [0]:
df_joined = spark.table("ecommerce_catalog.silver.cleaned_data")

####CREATING GOLD TABLE

####KPIs

In [0]:
df_gold = df_joined.select(
    "order_date",
    "product_id",
    "title",
    "category",
    "price",
    "total_quantity"
)

In [0]:
df_gold.display()

INFO:py4j.clientserver:Received command c on object id p0


order_date,product_id,title,category,price,total_quantity
2026-04-01,1,Essence Mascara Lash Princess,beauty,9.99,14
2026-04-01,2,Eyeshadow Palette with Mirror,beauty,19.99,4
2026-04-01,4,Red Lipstick,beauty,12.99,2
2026-04-01,5,Red Nail Polish,beauty,8.99,10
2026-04-01,6,Calvin Klein CK One,fragrances,49.99,6
2026-04-01,8,Dior J'adore,fragrances,89.99,6
2026-04-01,11,Annibale Colombo Bed,furniture,1899.99,10
2026-04-01,16,Apple,groceries,1.99,8
2026-04-01,18,Cat Food,groceries,8.99,8
2026-04-01,19,Chicken Meat,groceries,9.99,4


####1. Daily Revenue

In [0]:
#1. Daily revenue 

from pyspark.sql.functions import sum, col, to_date

def generate_daily_revenue(df):
    """
    Generate daily revenue KPI by summing (price * quantity) per day
    """
    try:
        logger.info("Generating Daily Revenue KPI")
        df_daily_revenue = df_gold.groupBy(
            to_date(col("order_date")).alias("order_date")
        ).agg(
            sum(col("price") * col("total_quantity")).alias("daily_revenue")
        )
    
        logger.info(f"Daily Revenue KPI computed successfully.")

        return df_daily_revenue
    except:
        logger.error(f"Error in Daily Revenue KPI")

df_daily_revenue = generate_daily_revenue(df_gold)
display(df_daily_revenue)

# Create external Delta table
df_daily_revenue.write.format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://ecommerce-data@staccecommerce.dfs.core.windows.net/Gold/Gold_Daily_Revenue") \
    .saveAsTable("ecommerce_catalog.gold.Gold_Daily_Revenue")


INFO:GoldLayer:Generating Daily Revenue KPI
INFO:GoldLayer:Daily Revenue KPI computed successfully.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


order_date,daily_revenue
2026-04-01,20512.059999999998


Databricks visualization. Run in Databricks to view.

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


####2. Top Selling Product

In [0]:
#2. Top-selling products 

from pyspark.sql.functions import sum, col

def generate_top_products(df):
    """
    Generate top-selling products KPI based on total quantity sold and revenue
    """
    try:
        logger.info("Generating Top Selling Products KPI")
        df_top_products = df_gold.groupBy(
            "product_id", "title", "category"
        ).agg(
            sum(col("total_quantity")).alias("total_quantity_sold"),
            sum(col("price") * col("total_quantity")).alias("total_revenue")
        ).orderBy(
            col("total_quantity_sold").desc()
        )

        logger.info(f"Top Selling Products KPI computed successfully.")

        return df_top_products
    except:
        logger.error(f"Error in Top Selling Products KPI")

df_top_products = generate_top_products(df_gold)
display(df_top_products)

df_top_products.write.format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://ecommerce-data@staccecommerce.dfs.core.windows.net/Gold/Gold_Top_Products") \
    .saveAsTable("ecommerce_catalog.gold.gold_top_products")


INFO:GoldLayer:Generating Top Selling Products KPI
INFO:GoldLayer:Top Selling Products KPI computed successfully.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


product_id,title,category,total_quantity_sold,total_revenue
1,Essence Mascara Lash Princess,beauty,14,139.86
20,Cooking Oil,groceries,10,49.900000000000006
11,Annibale Colombo Bed,furniture,10,18999.9
30,Kiwi,groceries,10,24.900000000000002
5,Red Nail Polish,beauty,10,89.9
16,Apple,groceries,8,15.92
18,Cat Food,groceries,8,71.92
24,Fish Steak,groceries,6,89.94
8,Dior J'adore,fragrances,6,539.9399999999999
6,Calvin Klein CK One,fragrances,6,299.94


Databricks visualization. Run in Databricks to view.

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


####3. Quantity Sold by Category

In [0]:
#3. Quantity Sold by Category

def generate_category_quantity(df):
    """
    Generate category-wise total quantity sold KPI
    """
    try:
        logger.info("Generating Category-wise total quantity sold KPI")

        df_category_quantity = df_gold.groupBy("category").agg(
            sum(col("total_quantity")).alias("total_quantity")
        ).orderBy(
            col("total_quantity").desc()
        )
        
        logger.info("Category-wise total quantity sold KPI computed successfully")

        return df_category_quantity
    except:
        logger.error(f"Error in Category-wise total quantity sold KPI")

df_category_quantity = generate_category_quantity(df_gold)
df_category_quantity.display()

df_category_quantity.write.format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://ecommerce-data@staccecommerce.dfs.core.windows.net/Gold/Gold_Category_Quantity") \
    .saveAsTable("ecommerce_catalog.gold.gold_category_quantity")


INFO:py4j.clientserver:Received command c on object id p1
INFO:GoldLayer:Generating Category-wise total quantity sold KPI
INFO:GoldLayer:Category-wise total quantity sold KPI computed successfully
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


category,total_quantity
groceries,62
beauty,30
fragrances,12
furniture,10


Databricks visualization. Run in Databricks to view.

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


####4. Revenue by Category

In [0]:
#4. Revenue by Category

from pyspark.sql.functions import sum, col

def generate_category_revenue(df):
    """
    Generate category-wise revenue KPI using price * quantity
    """
    try:
        logger.info("Generating Category-wise Revenue KPI")

        df_category_revenue = df.groupBy("category").agg(
            sum(col("price") * col("total_quantity")).alias("category_revenue")
        ).orderBy(
            col("category_revenue").desc()
        )
        
        logger.info("Category-wise Revenue KPI computed successfully")

        return df_category_revenue
    except:
        logger.error(f"Error in Category-wise Revenue KPI computation")

df_category_revenue = generate_category_revenue(df_gold)
df_category_revenue.display()

df_category_revenue.write.format("delta") \
    .mode("overwrite") \
    .option("path", "abfss://ecommerce-data@staccecommerce.dfs.core.windows.net/Gold/Gold_Category_Revenue") \
    .saveAsTable("ecommerce_catalog.gold.gold_category_revenue")


INFO:GoldLayer:Generating Category-wise Revenue KPI
INFO:GoldLayer:Category-wise Revenue KPI computed successfully
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


category,category_revenue
furniture,18999.9
fragrances,839.8799999999999
groceries,336.58
beauty,335.7


Databricks visualization. Run in Databricks to view.

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


####JDBC Connection

In [0]:
import os
jdbc_url = "jdbc:sqlserver://ecommerce--sql-server.database.windows.net:1433;database=ecommerce-database"

connection_properties = {
    "user": "ecommerce123",
    "password": os.getenv("DB_PASSWORD"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

####Loading KPIs to Azure SQL

In [0]:
df_daily_revenue.write.jdbc(
    url = jdbc_url,
    table = "gold_daily_revenue",
    mode = "overwrite",
    properties = connection_properties
)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


In [0]:
df_top_products.write.jdbc(
    url = jdbc_url,
    table = "gold_top_products",
    mode = "overwrite",
    properties = connection_properties
)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


In [0]:
df_category_quantity.write.jdbc(
    url = jdbc_url,
    table = "gold_category_qty",
    mode = "overwrite",
    properties = connection_properties
)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


In [0]:
df_category_revenue.write.jdbc(
    url = jdbc_url,
    table = "gold_category_revenue",
    mode = "overwrite",
    properties = connection_properties
)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
